# QC filtering 02 — Pal's labels and P = 0.14

Does Pal et al.'s per-sample QC remove cells at different rates in BRCA1+/- donors
and premenopausal WT donors? The donor is the unit of replication. This series
answers that in two phases. Phase 1 does not transfer labels.

| Notebook | Question |
|---|---|
| `00` cohort | Which 12 donors are in the comparison, and which analysis choices are frozen before any test? |
| `01` filtering rate | Does the removed fraction, overall and by reason, differ by genotype? |
| `02` Pal reference | Do the authors' own labels reproduce their retained-cell result (P = 0.14)? |
| `03` tipping point | How LP-enriched would BRCA1 removed cells have to be to hide an LP expansion? |

**This notebook uses Pal's own Fig 4B clusters.** It does not recluster the
cells and it does not test removed cells. The proportion model is a quasi-Poisson
regression of donor-by-cluster counts. The check passes only when the exported
cells are exactly the 59,766 retained cells and 0.14 lies inside the range of
four variants fixed in notebook 00: Pearson or deviance dispersion, and an F or
chi-square test of the cluster-by-genotype interaction.

It also freezes `c`, the largest near-threshold enrichment of any Fig 4B cluster.
The band is the 5% of each donor's retained cells closest to each of that donor's
own four QC thresholds. Enrichment pools all 12 donors and does not look at
genotype. That cluster value stays on record. Notebook 03 recomputes the same
band on Lim 2009 types and uses that ceiling for its zone.

The labels come from `SeuratObject_NormB1Total.rds` (Chen and Smyth, figshare
doi:10.6084/m9.figshare.17058077). Export them once with R, from the analysis
cache directory:

```
Rscript scripts/export_pal_norm_b1_labels.R \
  /path/to/SeuratObject_NormB1Total.rds \
  pal_norm_b1_labels.csv
```

The object was built with Seurat 3.1.1. The script needs that package. A Python
rebuild of the clustering is not this notebook.

In [ ]:
import logging

import pandas as pd
from IPython.display import Markdown, display

from signals_in_the_noise.analysis.pal_reference import (
    LABELS_FILENAME,
    load_pal_labels,
    run_pal_reference,
    write_pal_reference_outputs,
)
from signals_in_the_noise.analysis.qc_filtering import output_directory
from signals_in_the_noise.analysis.filtering_rate import load_phase1
from signals_in_the_noise.utils.logging_config import setup_logging

setup_logging(logging.INFO)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

OUTPUT_DIR = output_directory()
OUTPUT_DIR

## 1. Cohort and exported labels

Retained cells and per-sample thresholds come from notebook 00. Pal's cell names
are `N_0019_total_<barcode>` and `B1_0023_<barcode>`. `orig.ident` is only `N` or
`B1`, so the sample is taken from that prefix. `B1_0023` and `N_0123_total` are
different donors.

In [ ]:
phase1 = load_phase1(OUTPUT_DIR)
donors = pd.read_csv(OUTPUT_DIR / "donors.csv", index_col=0)
cells = pd.read_csv(OUTPUT_DIR / "cells.csv")
labels = load_pal_labels(OUTPUT_DIR / LABELS_FILENAME)
labels.shape

## 2. Reproduction and frozen enrichment

The handoff below is the text to paste back after this notebook runs.

In [ ]:
result = run_pal_reference(cells, donors, labels, phase1)
handoff = write_pal_reference_outputs(result, OUTPUT_DIR)
display(result["comparison"]["per_donor"])
display(result["variants"])
display(result["winners"])
display(Markdown(handoff))